# Approximating the Binomial Distribution

Two powerful approximations for $\text{Bin}(n, p)$:

1. **Normal approximation** — when $n$ is large and $p$ is not too extreme
2. **Poisson approximation** — when $n$ is large and $p$ is small

| Approximation | Condition | Result |
|---------------|-----------|--------|
| Normal | $np \geq 5$ and $n(1-p) \geq 5$ | $\text{Bin}(n,p) \approx N(np, np(1-p))$ |
| Poisson | $n \geq 20$, $p \leq 0.05$ | $\text{Bin}(n,p) \approx \text{Poi}(np)$ |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

print("Libraries loaded!")

---
## 1. Normal Approximation to Binomial (CLT)

By the Central Limit Theorem, the sum of many independent Bernoullis is approximately normal:

$$\text{Bin}(n, p) \approx N(\mu = np, \sigma^2 = np(1-p))$$

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

configs = [
    (10, 0.5, "n=10 (small n)"),
    (30, 0.5, "n=30 (moderate n)"),
    (100, 0.5, "n=100 (large n)"),
    (100, 0.1, "n=100, p=0.1 (skewed)"),
]

for ax, (n, p, title) in zip(axes.flat, configs):
    rv_bin = stats.binom(n, p)
    mu, var = n * p, n * p * (1 - p)
    rv_norm = stats.norm(mu, np.sqrt(var))
    
    x = np.arange(max(0, int(mu - 4*np.sqrt(var))), min(n, int(mu + 4*np.sqrt(var))) + 1)
    x_cont = np.linspace(x[0] - 1, x[-1] + 1, 300)
    
    ax.bar(x, rv_bin.pmf(x), color='#3498db', edgecolor='black', linewidth=0.8,
           alpha=0.7, label='Binomial (exact)')
    ax.plot(x_cont, rv_norm.pdf(x_cont), 'r-', linewidth=2.5,
            label=f'N({mu:.1f}, {var:.1f})')
    ax.set_title(f'{title}\nCheck: np={n*p:.1f}, n(1-p)={n*(1-p):.1f}',
                 fontsize=11, fontweight='bold')
    ax.set_xlabel('k')
    ax.set_ylabel('Probability / Density')
    ax.legend(fontsize=9)

plt.suptitle('Normal Approximation to Binomial (better as n grows)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 2. Continuity Correction

Since we're approximating a discrete distribution with a continuous one, we adjust by ±0.5:

$$P(X = k) \approx P(k - 0.5 < Y < k + 0.5)$$
$$P(X \leq k) \approx P(Y \leq k + 0.5)$$

where $Y \sim N(np, np(1-p))$.

In [ ]:
# Continuity correction example
n, p = 50, 0.4
rv_bin = stats.binom(n, p)
mu, sigma = n * p, np.sqrt(n * p * (1 - p))
rv_norm = stats.norm(mu, sigma)

k = 20
# Exact
p_exact = rv_bin.pmf(k)
# Without correction
p_no_corr = rv_norm.pdf(k)  # density at point (not really correct)
# With correction
p_with_corr = rv_norm.cdf(k + 0.5) - rv_norm.cdf(k - 0.5)

print(f"Bin({n}, {p}): P(X = {k})")
print(f"  Exact:                {p_exact:.6f}")
print(f"  Normal (no correction): {p_no_corr:.6f}")
print(f"  Normal (with ±0.5):    {p_with_corr:.6f}")
print()

# CDF comparison
k_cdf = 25
p_exact_cdf = rv_bin.cdf(k_cdf)
p_no_corr_cdf = rv_norm.cdf(k_cdf)
p_with_corr_cdf = rv_norm.cdf(k_cdf + 0.5)

print(f"Bin({n}, {p}): P(X ≤ {k_cdf})")
print(f"  Exact:                {p_exact_cdf:.6f}")
print(f"  Normal (no correction): {p_no_corr_cdf:.6f}")
print(f"  Normal (with +0.5):    {p_with_corr_cdf:.6f}")

# Visual
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(10, 35)
ax.bar(x, rv_bin.pmf(x), color='#3498db', edgecolor='black', alpha=0.6,
       width=1.0, label='Binomial PMF')
x_cont = np.linspace(10, 35, 300)
ax.plot(x_cont, rv_norm.pdf(x_cont), 'r-', linewidth=2.5, label='Normal approx')

# Highlight continuity correction for k=20
ax.fill_between([k-0.5, k+0.5], [0, 0],
                [rv_norm.pdf(k-0.5), rv_norm.pdf(k+0.5)],
                alpha=0.5, color='orange', label=f'±0.5 correction for k={k}')

ax.set_title(f'Continuity Correction: Bin({n}, {p})', fontsize=13, fontweight='bold')
ax.set_xlabel('k', fontsize=12)
ax.set_ylabel('Probability', fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

---
## 3. Poisson Approximation

When events are rare ($p$ small) and trials many ($n$ large):

$$\text{Bin}(n, p) \approx \text{Poi}(\lambda = np)$$

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

configs = [
    (20, 0.1, "n=20, p=0.1"),
    (100, 0.02, "n=100, p=0.02"),
    (1000, 0.003, "n=1000, p=0.003"),
]

for ax, (n, p, title) in zip(axes, configs):
    lam = n * p
    rv_bin = stats.binom(n, p)
    rv_poi = stats.poisson(lam)
    
    x = np.arange(0, int(lam + 4 * np.sqrt(lam)) + 2)
    
    ax.bar(x - 0.2, rv_bin.pmf(x), 0.4, color='#3498db', edgecolor='black',
           alpha=0.7, label=f'Bin({n}, {p})')
    ax.bar(x + 0.2, rv_poi.pmf(x), 0.4, color='#e74c3c', edgecolor='black',
           alpha=0.7, label=f'Poi({lam})')
    ax.set_title(f'{title}\nλ = np = {lam}', fontsize=11, fontweight='bold')
    ax.set_xlabel('k')
    ax.set_ylabel('P(X = k)')
    ax.legend(fontsize=9)

plt.suptitle('Poisson Approximation (better as n↑ and p↓)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 4. Comparison: When to Use Which?

In [ ]:
# Compare all three for a given scenario
n, p = 200, 0.03
lam = n * p
mu, sigma = n * p, np.sqrt(n * p * (1 - p))

rv_bin = stats.binom(n, p)
rv_poi = stats.poisson(lam)
rv_norm = stats.norm(mu, sigma)

x = np.arange(0, 20)

print(f"Bin({n}, {p}): np={lam}, n(1-p)={n*(1-p)}")
print(f"{'k':>3s} | {'Binomial':>10s} | {'Poisson':>10s} | {'Normal*':>10s}")
print("-" * 44)
for k in x:
    p_bin = rv_bin.pmf(k)
    p_poi = rv_poi.pmf(k)
    p_norm = rv_norm.cdf(k+0.5) - rv_norm.cdf(k-0.5)
    marker = ""
    if abs(p_poi - p_bin) < abs(p_norm - p_bin):
        marker = "← Poisson closer"
    else:
        marker = "← Normal closer"
    print(f"{k:3d} | {p_bin:10.6f} | {p_poi:10.6f} | {p_norm:10.6f}  {marker}")
print("* Normal uses continuity correction")

print(f"\nRule of thumb:")
print(f"  Use Poisson when: n ≥ 20, p ≤ 0.05, np ≤ 10")
print(f"  Use Normal when:  np ≥ 5 and n(1-p) ≥ 5")
print(f"  Here: np = {lam}, p = {p} → Poisson is better")

---
## Summary

| Approximation | When | Formula |
|---------------|------|---------|
| Normal | $np \geq 5$, $n(1-p) \geq 5$ | $\text{Bin}(n,p) \approx N(np, np(1-p))$ |
| Poisson | $n \geq 20$, $p \leq 0.05$ | $\text{Bin}(n,p) \approx \text{Poi}(np)$ |
| Continuity corr. | For normal approx | $P(X \leq k) \approx \Phi\left(\frac{k+0.5-\mu}{\sigma}\right)$ |

**Next:** Review notebook — formula reference for all distributions.